In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
#setting the random seed for reproducibility
torch.manual_seed(42)

In [3]:
df=pd.read_csv('/content/fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,pixel11,pixel12,pixel13,pixel14,pixel15,pixel16,pixel17,pixel18,pixel19,pixel20,pixel21,pixel22,pixel23,pixel24,pixel25,pixel26,pixel27,pixel28,pixel29,pixel30,pixel31,pixel32,pixel33,pixel34,pixel35,pixel36,pixel37,pixel38,pixel39,...,pixel745,pixel746,pixel747,pixel748,pixel749,pixel750,pixel751,pixel752,pixel753,pixel754,pixel755,pixel756,pixel757,pixel758,pixel759,pixel760,pixel761,pixel762,pixel763,pixel764,pixel765,pixel766,pixel767,pixel768,pixel769,pixel770,pixel771,pixel772,pixel773,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,125,72,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,3,0,2,0,0,210,228,228,233,0,0,0,0,0,0,0,0,0,31,81,133,184,201,190,117,0,0,2,1,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,43,117,34,15,24,33,117,80,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,51,144,152,...,157,158,161,148,159,58,0,6,0,0,0,0,0,0,0,0,0,4,0,60,143,143,148,146,152,152,148,148,147,145,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,0,0,0,2,0,33,114,37,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,...,11,17,13,13,11,10,7,5,5,5,7,0,0,0,1,0,0,41,69,88,86,94,106,114,118,47,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,0,2,0,58,145,114,10,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,...,165,153,155,134,143,172,215,62,0,0,0,0,0,0,0,0,10,190,178,194,209,211,209,205,211,215,213,217,225,228,213,203,174,151,188,10,0,0,0,0


In [4]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [7]:
#scaling the features
X_train=X_train/255.0
X_test=X_test/255.0

In [9]:
#creating the custom class
class CustomDataset(Dataset):
  def __init__(self,feature,label):
    self.feature=torch.tensor(feature,dtype=torch.float32)
    self.label=torch.tensor(label,dtype=torch.long)

  def __len__(self):
    return len(self.feature)

  def __getitem__(self,index):
    return self.feature[index],self.label[index]

In [11]:
train_dataset=CustomDataset(X_train,y_train)


In [13]:
len(train_dataset)

4800

In [14]:
test_dataset=CustomDataset(X_test,y_test)

In [15]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)#we dont shuffle during the training otherwise the accuracy wil keep on changing

In [20]:
#defing the model
class MyNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,x):
   return self.model(x)



In [21]:
epochs=100
learning_rate=0.1


In [22]:
model=MyNN(X_train.shape[1])

criterion=nn.CrossEntropyLoss()

optimizer=optim.SGD(model.parameters(),lr=learning_rate)

In [23]:
#training loop
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_features,batch_label in train_loader:

     #forward pass
     output=model(batch_features)

     #calculate the loss
     loss=criterion(output,batch_label)

     #zero the gradients
     optimizer.zero_grad()

     #backward pass
     loss.backward()

     #update the weights
     optimizer.step()

     total_epoch_loss=total_epoch_loss+loss.item()

  avg_loss=total_epoch_loss/len(train_loader)
  print(f'Epoch {epoch+1}/{epochs} Loss: {avg_loss:.4f}')



Epoch 1/100 Loss: 2.3037
Epoch 2/100 Loss: 2.3028
Epoch 3/100 Loss: 2.3026
Epoch 4/100 Loss: 2.3027
Epoch 5/100 Loss: 2.3027
Epoch 6/100 Loss: 2.3027
Epoch 7/100 Loss: 2.3026
Epoch 8/100 Loss: 2.3026
Epoch 9/100 Loss: 2.3026
Epoch 10/100 Loss: 2.3026
Epoch 11/100 Loss: 2.3024
Epoch 12/100 Loss: 2.3023
Epoch 13/100 Loss: 2.3026
Epoch 14/100 Loss: 2.3023
Epoch 15/100 Loss: 2.3024
Epoch 16/100 Loss: 2.3024
Epoch 17/100 Loss: 2.3022
Epoch 18/100 Loss: 2.3023
Epoch 19/100 Loss: 2.3023
Epoch 20/100 Loss: 2.3023
Epoch 21/100 Loss: 2.3020
Epoch 22/100 Loss: 2.3020
Epoch 23/100 Loss: 2.3021
Epoch 24/100 Loss: 2.3020
Epoch 25/100 Loss: 2.3017
Epoch 26/100 Loss: 2.3017
Epoch 27/100 Loss: 2.3014
Epoch 28/100 Loss: 2.3014
Epoch 29/100 Loss: 2.3014
Epoch 30/100 Loss: 2.3010
Epoch 31/100 Loss: 2.3009
Epoch 32/100 Loss: 2.3009
Epoch 33/100 Loss: 2.3004
Epoch 34/100 Loss: 2.2999
Epoch 35/100 Loss: 2.2998
Epoch 36/100 Loss: 2.2991
Epoch 37/100 Loss: 2.2990
Epoch 38/100 Loss: 2.2980
Epoch 39/100 Loss: 2.

In [24]:
#evaluation
model.eval()
total=0
correct=0
for batch_features,batch_label in test_loader:
  outputs=model(batch_features)
  _,predicted=torch.max(outputs,1) #  _ means ignore the valkues and give only the indices
  total+=batch_label.size(0)
  correct+=(predicted==batch_label).sum().item()
accuracy=correct/total
print(accuracy)

0.5208333333333334
